# TIGER (Toys and Games) — full pipeline on Kaggle

Ye notebook Amazon **Toys and Games** dataset par poora TIGER pipeline chalata hai:
content embeddings -> RQ-VAE -> Semantic IDs -> TIGER transformer -> evaluate -> Recall@K/Invalid@K plots.

Ye Beauty wale notebook ke bilkul parallel/independent hai — isko **alag Kaggle session** mein chalayein
(same session mein Beauty + Toys and Games + third dataset daalne se single-session GPU time-limit cross ho sakta hai).

**Pehle ye zaroor karein:**
1. `TIGER_Recommender_System_updated.zip` ko Kaggle par Private Dataset ke roop mein upload karein (agar pehle se nahi kiya).
2. Notebook Settings mein **Internet: ON**, **Accelerator: GPU** karein.
3. Us dataset ko is notebook mein **Add Data** se attach karein.

Expected dataset stats (paper Table 6): users=19,412 / items=11,924 / mean_seq_len=8.63 / median=6


In [ ]:
import os, shutil, glob

src_candidates = glob.glob("/kaggle/input/**/TIGER_Recommender_System", recursive=True)
assert src_candidates, "Project folder nahi mila — `!find /kaggle/input -maxdepth 6` chalakar check karein."
src = src_candidates[0]
dst = "/kaggle/working/tiger"

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
os.chdir(dst)
print("Working dir:", os.getcwd())
print(os.listdir("."))

In [ ]:
!pip install -q -r requirements.txt

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Stage A — Preprocess Amazon Toys and Games reviews -> leave-one-out sequences

Reviews dump download karta hai, 5-core filter apply karta hai, aur `data_toys/processed/{train,val,test}.jsonl` likhta hai.


In [ ]:
!python -m tiger.scripts.preprocess_category \
    --category Toys_and_Games \
    --output-dir data_toys/processed \
    --download

## Stage B — Item content embeddings (Sentence-T5)

Meta catalog download karta hai, Stage A ke sequence-items tak filter karta hai,
aur Sentence-T5 (base variant — XXL GPU memory mein fit nahi hota) se content embeddings banata hai.


In [ ]:
!python -m tiger.rqvae.dataloader_category \
    --config configs/rqvae/toys.yaml \
    --category Toys_and_Games \
    --download

## Stage C — Train RQ-VAE (paper hyperparameters)

Same architecture/hyperparameters jo Beauty ke liye use kiye the (paper-matched):
encoder `[512,256,128]` -> latent `32` -> 3-level residual quantizer, codebook size `256` per level,
Adagrad, batch=1024, 20000 epochs.

**Sanity check:** logs mein `util=[...]` (codebook utilization) dekhte rahein — target ≥80%.


In [ ]:
!python -m tiger.rqvae.train --config configs/rqvae/toys.yaml

## Stage D — Build item ↔ Semantic-ID lookup tables

In [ ]:
!python -m tiger.scripts.build_sid_tables \
    --checkpoint outputs/amazon_toys_checkpoints/best.pt \
    --items-csv  outputs/amazon_toys_items.csv \
    --embeddings outputs/amazon_toys_embeddings.npy \
    --sequences-dir data_toys/processed \
    --output-dir   data_toys

## Stage E — Train the TIGER transformer (full run, paper hyperparameters)

`d_model=128`, `6 heads`, `4+4 layers`, batch=256, Adafactor peak_lr=0.01 with 10k-step warmup + inverse-sqrt decay.
Paper trains Toys and Games for only **100,000 steps** (smaller dataset) — already set in the config. Early stopping (patience=5 evals) will stop earlier if validation NDCG@10 stops improving,
same as it did for Beauty.


In [ ]:
!python -m tiger.retrieval.train \
    --config configs/retrieval/toys.yaml \
    --data-dir data_toys \
    --output-dir outputs/tiger_toys

## Stage F — Evaluate the best checkpoint on the test split (vs. paper Table 1, Toys and Games)

In [ ]:
!python -m tiger.retrieval.evaluate \
    --checkpoint outputs/tiger_toys/checkpoints/best.pt \
    --data-dir   data_toys \
    --dataset    toys

## Stage G — Recall@K and Invalid-IDs@K plots (paper Figure 5/6 style)

In [ ]:
!python -m tiger.scripts.plot_recall_vs_k \
    --checkpoint outputs/tiger_toys/checkpoints/best.pt \
    --data-dir data_toys \
    --name toys \
    --ks 1,5,10,15,20 \
    --output-dir outputs/plots

In [ ]:
from IPython.display import Image, display
display(Image(f"outputs/plots/toys_recall_vs_k.png"))
display(Image(f"outputs/plots/toys_invalid_vs_k.png"))

## Save everything for download

In [ ]:
!zip -r /kaggle/working/tiger_toys_results.zip outputs data_toys/item_to_sid.json data_toys/sid_to_item.json data_toys/processed outputs/plots
print("Saved: /kaggle/working/tiger_toys_results.zip")